# DP2 Solar System Object Observations

Looks up a single Solar System object in the DP2 `SSObject` catalog by MPC designation (or `ssObjectId`), pulls every difference-image detection of it from the `DiaSource`/`SSSource` tables via the TAP service, and plots the light curve, phase curve, sky path, and heliocentric orbit position &mdash; using the same per-band colors/symbols and style conventions as the official DP2 tutorial notebooks (`lsst.utils.plotting`, `seaborn-v0_8-colorblind`).

Reference: [lsst/tutorial-notebooks, DP2/300_Science_demos/308_Solar_System](https://github.com/lsst/tutorial-notebooks/tree/main/DP2/300_Science_demos/308_Solar_System) and `DP2/200_Data_products/201_Catalogs` (`SSObject`, `SSSource`, `DiaSource` tables).

**Must run on the Rubin Science Platform** (data.lsst.cloud) with DP2 access &mdash; it queries the live TAP service, which isn't reachable from outside the RSP.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colormaps

from lsst.rsp import RSPDiscovery
from lsst.utils.plotting import get_multiband_plot_colors, get_multiband_plot_symbols

discovery = RSPDiscovery("dp2")
service = discovery.get_tap_client()

plt.style.use('seaborn-v0_8-colorblind')
filter_colors = get_multiband_plot_colors()
filter_names = list(filter_colors.keys())
filter_symbols = get_multiband_plot_symbols()

In [ ]:
def njy_to_ab_mag(flux_njy):
    """Convert a flux density in nanojansky to an AB apparent magnitude."""
    return -2.5 * np.log10(flux_njy / 3631e9)


def njy_to_ab_mag_err(flux_njy, flux_err_njy):
    """Propagate a flux error (nJy) into an AB magnitude error."""
    return (2.5 / np.log(10)) * (flux_err_njy / flux_njy)


def run_query(query):
    """Submit an ADQL query and return the result as an astropy table."""
    job = service.submit_job(query)
    job.run()
    job.wait(phases=['COMPLETED', 'ERROR'])
    if job.phase == 'ERROR':
        job.raise_if_error()
    assert job.phase == 'COMPLETED'
    results = job.fetch_result().to_table()
    job.delete()
    return results

## 2. Choose a target object

Set either `TARGET_DESIGNATION` (MPC designation, e.g. a numbered or provisional name) or `TARGET_SSOBJECT_ID` (the DP2 `ssObjectId`). Leave the one you're not using as `None`.

If the object hasn't been linked to an `ssObjectId` in DP2 yet, sections 3-8 (which need one) are skipped automatically &mdash; jump to Section 9, which doesn't.

In [ ]:
TARGET_DESIGNATION = "138P"
TARGET_SSOBJECT_ID = None

In [ ]:
designation = TARGET_DESIGNATION  # source of truth throughout, even if this object has no SSObject match yet

if TARGET_SSOBJECT_ID is not None:
    where_clause = f"sso.ssObjectId = {TARGET_SSOBJECT_ID}"
else:
    where_clause = f"sso.designation = '{TARGET_DESIGNATION}'"

sso_meta_cols = ['ssObjectId', 'designation', 'nObs', 'arc', 'MOIDEarth']
for b in filter_names:
    sso_meta_cols += [f'{b}_H', f'{b}_G12']

query = "SELECT " + ", ".join(f"sso.{c}" for c in sso_meta_cols) + \
        " FROM dp2.SSObject AS sso WHERE " + where_clause

sso_meta_rows = run_query(query)

if len(sso_meta_rows) > 0:
    sso_meta = sso_meta_rows[0]
    ss_object_id = int(sso_meta['ssObjectId'])
    designation = sso_meta['designation']
    print(f"designation:  {designation}")
    print(f"ssObjectId:   {ss_object_id}")
    print(f"nObs:         {sso_meta['nObs']}")
    print(f"arc (days):   {sso_meta['arc']}")
    print(f"MOID (au):    {sso_meta['MOIDEarth']}")
else:
    sso_meta = None
    ss_object_id = None
    print(f"No SSObject linked yet for {designation!r} -- sections 3-8 need a linked "
          f"ssObjectId and will be skipped. Section 9 doesn't need one -- it searches "
          f"DiaSource directly.")

## 3. Query all observations

Join `DiaSource` (per-detection photometry) to `SSSource` (per-detection heliocentric/topocentric geometry) for every detection of this `ssObjectId`.

In [ ]:
obs = None
if ss_object_id is not None:
    query = f"""
    SELECT dias.midpointMjdTai, dias.band, dias.ra, dias.dec,
           dias.psfFlux, dias.psfFluxErr,
           sss.topoRange, sss.helioRange, sss.phaseAngle,
           sss.helio_x, sss.helio_y
    FROM dp2.DiaSource AS dias
    JOIN dp2.SSSource AS sss ON dias.diaSourceId = sss.diaSourceId
    WHERE dias.ssObjectId = {ss_object_id}
    ORDER BY dias.midpointMjdTai ASC
    """
    obs = run_query(query)
    print(f"Retrieved {len(obs)} observations of {designation}")

## 4. Convert flux to magnitude

In [ ]:
mag = mag_err = reduced_mag = None
if obs is not None:
    obs = obs[obs['psfFlux'] > 0]
    mag = njy_to_ab_mag(obs['psfFlux'])
    mag_err = njy_to_ab_mag_err(obs['psfFlux'], obs['psfFluxErr'])
    reduced_mag = mag - 5 * np.log10(obs['helioRange'] * obs['topoRange'])

## 5. Light curve

Apparent PSF magnitude vs. time, one color/symbol per band (Rubin's standard *ugrizy* scheme).

In [ ]:
if obs is None:
    print(f"Skipping -- no linked ssObjectId for {designation!r}")
else:
    fig, ax = plt.subplots(figsize=(9, 5))

    for filt in filter_names:
        fx = obs['band'] == filt
        if fx.any():
            ax.errorbar(obs['midpointMjdTai'][fx], mag[fx], yerr=mag_err[fx],
                        fmt=filter_symbols[filt], color=filter_colors[filt],
                        ms=6, mew=0, alpha=0.8, ecolor=filter_colors[filt],
                        elinewidth=1, capsize=2, label=filt)

    ax.invert_yaxis()
    ax.set_xlabel('MJD (TAI)')
    ax.set_ylabel('PSF magnitude (AB)')
    ax.set_title(f'Light curve: {designation}')
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()

## 6. Phase curve

Reduced magnitude (apparent magnitude corrected to 1 au from the Sun and 1 au from the observer) vs. phase angle. Dashed lines mark each band's fitted absolute magnitude `[band]_H` from the `SSObject` table, which is the reduced magnitude extrapolated to phase angle 0.

In [ ]:
if obs is None:
    print(f"Skipping -- no linked ssObjectId for {designation!r}")
else:
    fig, ax = plt.subplots(figsize=(7, 5))

    for filt in filter_names:
        fx = obs['band'] == filt
        if fx.any():
            ax.plot(obs['phaseAngle'][fx], reduced_mag[fx],
                    filter_symbols[filt], color=filter_colors[filt],
                    ms=6, mew=0, alpha=0.8, label=filt)
            h_mag = sso_meta[f'{filt}_H']
            if h_mag is not None and not np.isnan(h_mag):
                ax.axhline(h_mag, color=filter_colors[filt], ls='--', lw=1, alpha=0.6)

    ax.invert_yaxis()
    ax.set_xlabel('Phase angle (deg)')
    ax.set_ylabel('Reduced magnitude (mag, at 1 au)')
    ax.set_title(f'Phase curve: {designation}')
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()

## 7. Sky path

Right ascension and declination of every detection, colored by observation date.

In [ ]:
if obs is None:
    print(f"Skipping -- no linked ssObjectId for {designation!r}")
else:
    fig, ax = plt.subplots(figsize=(9, 4))
    im = ax.scatter(obs['ra'], obs['dec'], c=obs['midpointMjdTai'],
                    cmap=colormaps['viridis'], s=20)
    ax.set_xlabel('ra (deg)')
    ax.set_ylabel('dec (deg)')
    ax.set_title(f'Sky path: {designation}')
    ax.grid(True, alpha=0.3)
    fig.colorbar(im, ax=ax, label='MJD (TAI)')
    plt.tight_layout()
    plt.show()

## 8. Heliocentric orbit position

Projection of the object's heliocentric X/Y position at the time of every observation, with the Sun at the origin.

In [ ]:
if obs is None:
    print(f"Skipping -- no linked ssObjectId for {designation!r}")
else:
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(obs['helio_x'], obs['helio_y'], 'o', ms=4, mew=0, color='black')
    ax.plot(0, 0, '*', ms=15, color='darkorange', label='Sun')
    ax.set_xlabel('heliocentric X (au)')
    ax.set_ylabel('heliocentric Y (au)')
    ax.set_title(f'Heliocentric position: {designation}')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 9. Precovery: candidates beyond the DP2 linkage

DP2's `SSObject`/`SSSource` linkage only tags a `DiaSource` with an `ssObjectId` once Rubin's own Solar System Processing successfully links it to an orbit. Real detections of this object can still be sitting in the raw `DiaSource` table unlinked &mdash; e.g. epochs before the orbit was well-determined enough to link, or a cometary non-gravitational orbit that defeats the standard linker.

Rather than predict the object's position from an orbit, this section uses observations Rubin (station `X05`) has actually reported to the Minor Planet Center for the object &mdash; pulled from the bulk MPC/SBN observation archive with `scripts/extract_mpc_x05_observations.py` &mdash; and cross-matches those real positions/times directly against `dp2.DiaSource`, independent of `ssObjectId`. Because these are confirmed reported observations rather than a predicted ephemeris, matches are much more trustworthy than a probabilistic search, though duplicate or spurious matches within the search tolerance are still possible &mdash; treat this as a strong candidate list, not a final answer.

**To target a different object**, on your own machine (where the bulk archive lives):

```bash
python scripts/extract_mpc_x05_observations.py "designation"
```

then `git add`/`commit`/`push` the resulting `data/precovery/<designation>_x05_obs.csv`, `git pull` it down here, and update `MPC_OBS_CSV_PATH` below to match.

In [ ]:
from astropy.table import Table

MPC_OBS_CSV_PATH = "../data/precovery/138P_x05_obs.csv"  # relative to notebooks/, from scripts/extract_mpc_x05_observations.py
PRECOVERY_SEARCH_RADIUS_DEG = 0.003                       # ~11 arcsec
PRECOVERY_TIME_TOL_DAYS = 0.01                            # ~14 min, to bracket one exposure without reaching the next visit

### 9.1. Load MPC's reported X05 observations

These are real observations Rubin reported to the MPC for this object, not a predicted ephemeris.

In [ ]:
mpc_obs = Table.read(MPC_OBS_CSV_PATH)
print(f"{len(mpc_obs)} MPC-reported X05 observations of {designation}")

### 9.2. Cross-match against the raw DiaSource table

Upload the MPC observation table and cross-match it against `dp2.DiaSource` directly by position and time &mdash; not joined through `SSObject`/`SSSource` &mdash; so unlinked detections are included.

In [ ]:
query = f"""
SELECT dias.diaSourceId, dias.ssObjectId, dias.visit, dias.band,
       dias.midpointMjdTai, dias.ra, dias.dec,
       dias.psfFlux, dias.psfFluxErr,
       mpc.mjd AS mpc_mjd, mpc.mag AS mpc_mag, mpc.band AS mpc_band
FROM dp2.DiaSource AS dias
JOIN TAP_UPLOAD.mpc_obs AS mpc
ON DISTANCE(POINT('ICRS', dias.ra, dias.dec),
            POINT('ICRS', mpc.ra, mpc.dec)) < {PRECOVERY_SEARCH_RADIUS_DEG}
AND ABS(dias.midpointMjdTai - mpc.mjd) < {PRECOVERY_TIME_TOL_DAYS}
"""
job = service.submit_job(query, uploads={"mpc_obs": mpc_obs})
job.run()
job.wait(phases=['COMPLETED', 'ERROR'])
if job.phase == 'ERROR':
    job.raise_if_error()
assert job.phase == 'COMPLETED'
precovery = job.fetch_result().to_table()
job.delete()

print(f"{len(precovery)} candidate detections matching MPC-reported positions/times")

In [ ]:
new_candidates = None
if precovery is not None and len(precovery) > 0:
    if ss_object_id is not None:
        already_linked = precovery['ssObjectId'] == ss_object_id
        new_candidates = precovery[~already_linked]
        print(f"{already_linked.sum()} already linked to this ssObjectId, "
              f"{len(new_candidates)} unlinked candidates")
    else:
        new_candidates = precovery
        print(f"{len(new_candidates)} candidates (no confirmed ssObjectId for {designation!r} "
              f"to check against -- some of these rows may already be linked to some "
              f"SSObject, just not one matched earlier by designation string)")

    new_candidates = new_candidates[new_candidates['psfFlux'] > 0]
    candidate_mag = njy_to_ab_mag(new_candidates['psfFlux'])

### 9.3. Plot: light curve with precovery candidates

Same light curve as Section 5, with the unlinked candidates overlaid as hollow markers in the matching band color.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

if obs is not None:
    for filt in filter_names:
        fx = obs['band'] == filt
        if fx.any():
            ax.errorbar(obs['midpointMjdTai'][fx], mag[fx], yerr=mag_err[fx],
                        fmt=filter_symbols[filt], color=filter_colors[filt],
                        ms=6, mew=0, alpha=0.8, ecolor=filter_colors[filt],
                        elinewidth=1, capsize=2, label=filt)

if new_candidates is not None and len(new_candidates) > 0:
    for filt in filter_names:
        fx = new_candidates['band'] == filt
        if fx.any():
            ax.plot(new_candidates['midpointMjdTai'][fx], candidate_mag[fx],
                    filter_symbols[filt], mfc='none', mec=filter_colors[filt],
                    ms=8, mew=1.5, alpha=0.9)
    ax.plot([], [], 'o', mfc='none', mec='black', mew=1.5, label='unlinked candidate')

ax.invert_yaxis()
ax.set_xlabel('MJD (TAI)')
ax.set_ylabel('PSF magnitude (AB)')
ax.set_title(f'Light curve with precovery candidates: {designation}')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

## 10. Alert cutouts from Fink

Every Rubin alert bundles three cutouts &mdash; the current (science) exposure, the template used for image differencing, and the difference image itself. [Fink](https://fink-broker.org) ingests that alert stream and exposes those cutouts by `diaSourceId` through a dedicated LSST API (`api.lsst.fink-portal.org`).

DP2's `diaSourceId` comes from *annual Data Release* reprocessing &mdash; a separate pipeline run from the *Prompt Processing* that generates the live alert stream Fink ingests &mdash; so a DP2 `diaSourceId` isn't guaranteed to be the same one Fink has cutouts under. `get_alert_cutouts` below tries it directly first, and falls back to a Fink cone search (position + a tight time window) to resolve the right ID when that misses.

In [ ]:
import io
import requests
import pandas as pd
from astropy.io import fits
from astropy.time import Time

FINK_BASE_URL = "https://api.lsst.fink-portal.org/api/v1"
FINK_CONE_RADIUS_ARCSEC = 2.0
FINK_TIME_WINDOW_DAYS = 0.01  # ~14 min, same rationale as the DP2 cross-match tolerance in section 9
CUTOUT_KINDS = ["Science", "Template", "Difference"]

In [ ]:
def fetch_fink_cutout(dia_source_id, kind):
    """Fetch one cutout (Science/Template/Difference) as a FITS HDUList, or None if not found."""
    if dia_source_id is None:
        return None
    r = requests.post(f"{FINK_BASE_URL}/cutouts", json={
        "diaSourceId": str(dia_source_id),
        "kind": kind,
        "output-format": "FITS",
    })
    try:
        return fits.open(io.BytesIO(r.content))
    except OSError:
        return None


def resolve_fink_dia_source_id(ra, dec, mjd):
    """Cone search Fink for the alert nearest (ra, dec) within a tight time window; return its diaSourceId."""
    r = requests.post(f"{FINK_BASE_URL}/conesearch", json={
        "ra": str(ra),
        "dec": str(dec),
        "radius": str(FINK_CONE_RADIUS_ARCSEC),
        "startdate": Time(mjd - FINK_TIME_WINDOW_DAYS, format='mjd').iso,
        "stopdate": Time(mjd + FINK_TIME_WINDOW_DAYS, format='mjd').iso,
        "kind": "within",
        "columns": "r:diaSourceId",
    })
    pdf = pd.read_json(io.BytesIO(r.content))
    return pdf["r:diaSourceId"].iloc[0] if len(pdf) > 0 else None


def get_alert_cutouts(dia_source_id, ra, dec, mjd):
    """Get the Science/Template/Difference cutouts for one detection.

    Tries dia_source_id directly against Fink; falls back to resolving the
    Fink-native diaSourceId via a position + time cone search if that misses.
    """
    cutouts = {kind: fetch_fink_cutout(dia_source_id, kind) for kind in CUTOUT_KINDS}
    if all(c is None for c in cutouts.values()):
        resolved_id = resolve_fink_dia_source_id(ra, dec, mjd)
        if resolved_id is not None:
            cutouts = {kind: fetch_fink_cutout(resolved_id, kind) for kind in CUTOUT_KINDS}
    return cutouts

### 10.1. Example: pull cutouts for one candidate

Takes the brightest section 9 candidate (falls back to the brightest linked observation if there are no candidates) and fetches its cutouts.

In [ ]:
example = None
example_dia_source_id = None

if new_candidates is not None and len(new_candidates) > 0:
    example = new_candidates[np.argmax(new_candidates['psfFlux'])]
    example_dia_source_id = example['diaSourceId']
elif obs is not None and len(obs) > 0:
    example = obs[np.argmax(obs['psfFlux'])]  # section 3's query didn't select diaSourceId

if example is not None:
    cutouts = get_alert_cutouts(example_dia_source_id, example['ra'], example['dec'], example['midpointMjdTai'])
    print(f"ra={example['ra']:.5f}  dec={example['dec']:.5f}  MJD={example['midpointMjdTai']:.3f}")
    for kind in CUTOUT_KINDS:
        print(f"  {kind}: {'found' if cutouts[kind] is not None else 'not found'}")
else:
    cutouts = None
    print("No observations or candidates available to fetch cutouts for.")

In [ ]:
if cutouts is not None:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, kind in zip(axes, CUTOUT_KINDS):
        hdul = cutouts[kind]
        if hdul is None:
            ax.set_title(f"{kind} (not found)")
            ax.axis('off')
            continue
        image = hdul[0].data.astype(float)
        vmax = np.percentile(np.abs(image), 99.5)
        im = ax.imshow(image, origin='lower', cmap='gray', vmin=-0.2 * vmax, vmax=vmax)
        fig.colorbar(im, ax=ax)
        ax.set_title(kind)

    fig.suptitle(f"{designation}: alert cutouts near ({example['ra']:.4f}, {example['dec']:.4f}), "
                 f"MJD {example['midpointMjdTai']:.3f}")
    plt.tight_layout()
    plt.show()